In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys

sys.path.append("../../existing_metric_comparison/")
sys.path.append("../../../")
sys.path.append("../../utils/")
import data_cleanup
import metrics_lib as metrics
import os
import mapqc

In [ ]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

## Set global variables

In [ ]:
study = "Sheppard_2020"
batch_variable = "sample"
kl = "1.0"
use_old_counts = True
distance_type = "pairwise_euclidean"
if use_old_counts == True:
    suffix = "original_counts_"
else:
    suffix = "cellxgene_counts_"

path_emb = f"/lustre/groups/ml01/workspace/lisa.sikkema/atlas_extension/data/HLCA/HLCA_and_{study}_{suffix}batch_{batch_variable}_kl{kl}_latent.h5ad"
path_counts = f"/lustre/groups/ml01/workspace/lisa.sikkema/atlas_extension/data/HLCA/HLCA_and_{study}_counts.h5ad"
path_results_dir = f"/lustre/groups/ml01/code/lisa.sikkema/atlas_extension/results/HLCA/{study}/{suffix}batch_{batch_variable}_kl{kl}/benchmark"
path_mapqc_results_dir = f"/lustre/groups/ml01/code/lisa.sikkema/atlas_extension/results/HLCA/{study}/{suffix}batch_{batch_variable}_kl{kl}/mapqc"

In [ ]:
output_file_path_prefix = f"/lustre/groups/ml01/code/lisa.sikkema/atlas_extension/results/downsampling_query/{study}_{suffix}batch_{batch_variable}_kl{kl}_{distance_type}_"

In [ ]:
output_file_path_prefix

In [ ]:
if not os.path.exists(path_results_dir):
    if os.path.exists(os.path.dirname(path_results_dir)):
        print(f"Creating directory {path_results_dir}")
        os.mkdir(RESULT_DIR)
    else:
        print("Parent directory of path_results_dir does not exist. Create it first.")

In [ ]:
path_emb = "/lustre/groups/ml01/workspace/lisa.sikkema/atlas_extension/data/HLCA/HLCA_and_Sheppard_2020_original_counts_batch_sample_kl1.0_latent.h5ad"

In [ ]:
path_counts = "/lustre/groups/ml01/workspace/lisa.sikkema/atlas_extension/data/HLCA/HLCA_and_Sheppard_2020_counts.h5ad"

In [ ]:
n_repeats = 3

## load and prep data

In [ ]:
adata_emb = sc.read_h5ad(path_emb)
adata_counts = sc.read_h5ad(path_counts)
adata_full = data_cleanup.add_mapping_info_to_counts_adata(adata_counts, adata_emb)

### subset to stromal compartment, and healthy only:

In [ ]:
sc.pl.umap(adata_full, color="leiden", legend_loc="on data")

In [ ]:
adata_full.shape

In [ ]:
stroma = ["5", "10"]
EC = ["7", "16"]

In [ ]:
adata = adata_full[adata_full.obs.leiden.isin(stroma), :].copy()

In [ ]:
adata.shape

In [ ]:
adata = adata[adata.obs.lung_condition != "IPF", :].copy()

In [ ]:
adata.shape

### Run metrics without downsampling:

In [ ]:
# lisi_batch_results_no_downsample = dict()

In [ ]:
# for cycle in n_repeats:
#     lisi_batch_results_no_downsample[cycle] = metrics.calculate_scib_metrics(
#         adata=adata,
#         obsm_emb_key="X_scanvi_emb",
#         batch_key="r_or_q",
#         dataset_key="r_or_q",
#         sample_key="sample",
#         dry_run=False,
#     )["lisi_batch"]["per_cell"].values

In [ ]:
# # make sure k etc. are the same as in the full run, for fair comparison
# mapqc_results_no_downsample = mapqc.MapqcResults(
#     adata=adata,
#     ref_q_key="r_or_q",
#     ref_cat="r",
#     query_cat="q",
#     condition_key="status",
#     control_cats="control",
#     control_or_case="control",
#     case_cats=[],
#     emb_key="X_scanvi_emb",
#     dataset_key="study",
#     sample_key="sample",
#     sampling_group_cat="leiden",
#     n_cores=2,
#     distance_type="edistance",
#     k=2924,
#     max_k=3508,
#     n_nhoods=50,  # change this to not hardcoded, for fair comparison. But for now doing this for a quick check
# )
# mapqc_results_no_downsample.run_mapqc()
# mapqc_scores_no_downsample = mapqc_results_no_downsample.adata.obs.mapqc_score.dropna()

## Run metrics with downsampling of query

In [ ]:
downsample_fractions = [1, 0.5, 0.1, 0.01]

In [ ]:
query_cells = adata.obs.index[adata.obs.r_or_q == "q"].tolist()

In [ ]:
lisi_batch_results = dict()
mapqc_results = dict()
query_subsets = dict()

In [ ]:
for frac in downsample_fractions:
    for cycle in range(0, n_repeats):
        query_subset = np.random.choice(
            query_cells, size=int(frac * len(query_cells)), replace=False
        )
        adata_subset = adata[
            (adata.obs.r_or_q == "r") | adata.obs.index.isin(query_subset), :
        ].copy()
        lisi_batch_results[f"{frac} ({cycle})"] = metrics.calculate_scib_metrics(
            adata=adata_subset,
            obsm_emb_key="X_scanvi_emb",
            batch_key="r_or_q",
            dataset_key="r_or_q",
            sample_key="sample",
            dry_run=False,
        )["lisi_batch"]["per_cell"].values
        # make sure k etc. are the same as in the full run, for fair comparison
        mapqc_results_downsample = mapqc.MapqcResults(
            adata=adata_subset,
            ref_q_key="r_or_q",
            ref_cat="r",
            query_cat="q",
            condition_key="status",
            control_cats="control",
            control_or_case="control",
            case_cats=[],
            emb_key="X_scanvi_emb",
            dataset_key="study",
            sample_key="sample",
            sampling_group_cat="leiden",
            n_cores=2,
            distance_type=distance_type,
            k=2924,
            max_k=3508,
            n_nhoods=50,  # change this to not hardcoded, for fair comparison. But for now doing this for a quick check
        )
        mapqc_results_downsample.run_mapqc()
        mapqc_results[
            f"{frac} ({cycle})"
        ] = mapqc_results_downsample.adata.obs.mapqc_score.dropna()
        query_subsets[f"{frac} ({cycle})"] = query_subset

## Plot results of different runs

### batch lisi:

In [ ]:
results_df_lisi = pd.DataFrame()

In [ ]:
for frac in downsample_fractions:
    for cycle in range(0, n_repeats):
        results_df_lisi = pd.concat(
            (
                results_df_lisi,
                pd.DataFrame(
                    {
                        "lisi_score": lisi_batch_results[f"{frac} ({cycle})"],
                        "run": f"downsample {frac} ({cycle})",
                    }
                ),
            )
        )

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(results_df_lisi, x="run", y="lisi_score", ax=ax, color="grey", fliersize=0)
# sns.stripplot(results_df_lisi, x="run", y="lisi_score", ax=ax, size=0.1, color="black")
ax.tick_params(axis="x", rotation=90)
plt.show()

### mapqc:

In [ ]:
results_df_mapqc = pd.DataFrame()  # columns=["lisi_score", "run"])

In [ ]:
for frac in downsample_fractions:
    for cycle in range(0, n_repeats):
        results_df_mapqc = pd.concat(
            (
                results_df_mapqc,
                pd.DataFrame(
                    {
                        "mapqc_score": mapqc_results[f"{frac} ({cycle})"].values,
                        "run": f"downsample {frac} ({cycle})",
                    }
                ),
            )
        )

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(
    results_df_mapqc, x="run", y="mapqc_score", ax=ax, color="grey", fliersize=0
)
sns.stripplot(
    results_df_mapqc, x="run", y="mapqc_score", ax=ax, size=0.1, color="black"
)
ax.tick_params(axis="x", rotation=90)
plt.show()

In [ ]:
# sns.violinplot(results_df_mapqc, x="run", y="mapqc_score")

### Store results:

In [ ]:
results_df_mapqc.to_csv(f"{output_file_path_prefix}mapqc_scores.csv")
results_df_lisi.to_csv(f"{output_file_path_prefix}lisi_batch_scores.csv")